In [1]:
# Force TF backend, fix seeds so the sweep is reproducible, and stay on CPU.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import tensorflow as tf
from ee446_data import (
    load_export, make_windows, group_train_test_split, summarize,
    TARGET_LABELS, WINDOW_SAMPLES,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.set_visible_devices([], "GPU")   # CPU only: keeps results deterministic

print("TF:", tf.__version__, "| NumPy:", np.__version__)

TF: 2.14.1 | NumPy: 1.26.4


In [2]:
# Two exports: the original one every model trained on, and a later export that
# also contains recordings made after training. Inspect one record's structure.
recs_old = load_export("ee-446-final-project-export")
recs_new = load_export("ei-export-v2")

r = recs_new[0]
print(type(r))
print(r if not hasattr(r, "keys") else {k: str(v)[:60] for k, v in r.items()})
print("old:", len(recs_old), "new:", len(recs_new))

Found 1033 sample files under ee-446-final-project-export  (1033 .cbor, 0 .json)

Loaded 1033 samples across 13 labels:
  Circle         61
  Down           60
  Down1         120
  Idle           91
  Left           64
  Left1         120
  Right          61
  Right1        125
  Square         76
  Triangle       60
  Up             60
  Up1           120
  testing        15
Found 1084 sample files under ei-export-v2  (1084 .cbor, 0 .json)

Loaded 1084 samples across 13 labels:
  Circle         61
  Down           60
  Down1         130
  Idle          101
  Left           64
  Left1         130
  Right          61
  Right1        136
  Square         76
  Triangle       60
  Up             60
  Up1           130
  testing        15
<class 'dict'>
{'label': 'Circle', 'sample_id': 'Circle.6vm62pdt.ingestion-8489d5465f-lk8b9', 'device': '40:AA:6C:2E:91:F1', 'values': '[[-0.736217   -2.307412    9.385869    0.88500977 -4.91333  ', 'interval_ms': '10.0', 'path': 'ei-export-v2/testing/Cir

In [3]:
# Diff by sample_id to find the 51 recordings that exist only in the newer export.
# These were collected on the device AFTER training, so no model has seen them.
old_ids = {r["sample_id"] for r in recs_old}
new_recs = [r for r in recs_new if r["sample_id"] not in old_ids]
new_active = [r for r in new_recs if r["label"] in TARGET_LABELS]

print("new recordings:", len(new_recs), "| active-class:", len(new_active))
from collections import Counter
print("labels:", Counter(r["label"] for r in new_active))
print("devices:", Counter(r["device"] for r in new_active))

# Split 60/40 at the recording level: 31 recordings to adapt on, 20 to evaluate on.
Xn, yn, gn, classes_n = make_windows(new_active, TARGET_LABELS)
Xa_tr, ya_tr, Xa_ev, ya_ev = group_train_test_split(Xn, yn, gn, test_size=0.4, seed=SEED)
summarize(ya_tr, classes_n, "adapt")
summarize(ya_ev, classes_n, "eval")

new recordings: 51 | active-class: 51
labels: Counter({'Right1': 11, 'Down1': 10, 'Idle': 10, 'Left1': 10, 'Up1': 10})
devices: Counter({'75:28:4B:21:F1:4F': 51})
Windows: (306, 150, 6)  (window=150, stride=10, scale=0.05)
  Down1           60
  Idle            60
  Left1           60
  Right1          66
  Up1             60
Recordings: 51  ->  train 31 / test 20
Windows:    train 186 / test 120
adapt: 186 windows
  Down1           42  (22.6%)
  Idle            48  (25.8%)
  Left1           24  (12.9%)
  Right1          42  (22.6%)
  Up1             30  (16.1%)
eval: 120 windows
  Down1           18  (15.0%)
  Idle            12  (10.0%)
  Left1           36  (30.0%)
  Right1          24  (20.0%)
  Up1             30  (25.0%)


In [4]:
# Recreate the exact train/test split used in Compress.ipynb (same seed), so the
# original test accuracy can be compared before and after adaptation.
Xt, yt, gt, target_classes = make_windows(recs_old, TARGET_LABELS)
Xt_tr, yt_tr, Xt_te, yt_te = group_train_test_split(Xt, yt, gt, test_size=0.2, seed=SEED)

base = tf.keras.models.load_model("models/minimal_1conv.h5")
base.compile(optimizer=tf.keras.optimizers.legacy.Adam(1e-4),
             loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 0.9928 confirms the split reproduced correctly.
_, acc_orig = base.evaluate(Xt_te, yt_te, verbose=0)
_, acc_new_before = base.evaluate(Xa_ev, ya_ev, verbose=0)
print(f"original test set:      {acc_orig:.4f}   (expect 0.9928)")
print(f"new recordings, before: {acc_new_before:.4f}")

Windows: (3456, 150, 6)  (window=150, stride=10, scale=0.05)
  Down1          720
  Idle           546
  Left1          720
  Right1         750
  Up1            720
Recordings: 576  ->  train 461 / test 115
Windows:    train 2766 / test 690
original test set:      0.9928   (expect 0.9928)
new recordings, before: 1.0000


In [5]:
# Same unseen recordings, every model: does model size affect generalization?
# Larger models can memorize training-recording quirks; smaller ones cannot.
from tensorflow.keras.optimizers.legacy import Adam

names = ["full_16_32_64", "half_8_16_32", "quarter_4_8_16",
         "half_pruned50", "eighth_2_4_8", "minimal_1conv", "scratch_5class"]

print(f"{'model':16s} {'orig test':>9s} {'new eval':>9s}")
for n in names:
    m = tf.keras.models.load_model(f"models/{n}.h5")
    m.compile(optimizer=Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    _, a_old = m.evaluate(Xt_te, yt_te, verbose=0)
    _, a_new = m.evaluate(Xa_ev, ya_ev, verbose=0)
    print(f"{n:16s} {a_old:>9.4f} {a_new:>9.4f}")

model            orig test  new eval
full_16_32_64       0.9884    0.9583
half_8_16_32        0.9899    0.9750
quarter_4_8_16      0.9928    0.9750
half_pruned50       0.9884    0.9500
eighth_2_4_8        0.8913    0.7667
minimal_1conv       0.9928    1.0000
scratch_5class      0.9884    0.9500


In [6]:
# The full-width CNN dropped the most on unseen recordings, so it is the one
# worth adapting. Fine-tuning = continue training the trained model briefly at a
# low learning rate (1e-4) on 31 of the new recordings.
ft = tf.keras.models.load_model("models/full_16_32_64.h5")
ft.compile(optimizer=Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

_, before_new = ft.evaluate(Xa_ev, ya_ev, verbose=0)
_, before_old = ft.evaluate(Xt_te, yt_te, verbose=0)

ft.fit(Xa_tr, ya_tr, epochs=25, batch_size=16, verbose=0)

# Two measurements: did it improve on new data, and did it forget the old data?
_, after_new = ft.evaluate(Xa_ev, ya_ev, verbose=0)
_, after_old = ft.evaluate(Xt_te, yt_te, verbose=0)
ft.save("models/full_finetuned_newdata.h5")

print(f"new-data eval:     {before_new:.4f} -> {after_new:.4f}")
print(f"original test set: {before_old:.4f} -> {after_old:.4f}   (forgetting check)")

new-data eval:     0.9583 -> 0.9833
original test set: 0.9884 -> 0.9855   (forgetting check)


/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [7]:
# Same INT8 PTQ as in the compression study, applied to the fine-tuned model,
# to confirm quantization stays lossless after adaptation.
def to_tflite_int8(model):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    def rep_data():
        for i in range(0, len(Xt_tr), 25):
            yield [Xt_tr[i:i+1].astype(np.float32)]
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep_data
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    return conv.convert()

def eval_tflite(tfl, X, y):
    # Quantize each input the way the device would, then run the interpreter.
    it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    c = 0
    for i in range(len(X)):
        x = X[i:i+1].astype(np.float32)
        s, z = inp["quantization"]
        x = (x/s + z).round().clip(-128,127).astype(np.int8)
        it.set_tensor(inp["index"], x); it.invoke()
        c += int(it.get_tensor(out["index"])[0].argmax() == y[i])
    return c/len(X)

tfl = to_tflite_int8(ft)
open("models/full_finetuned_newdata_int8.tflite","wb").write(tfl)
print(f"int8 new-data eval:     {eval_tflite(tfl, Xa_ev, ya_ev):.4f}   (fp32: 0.9833)")
print(f"int8 original test set: {eval_tflite(tfl, Xt_te, yt_te):.4f}   (fp32: 0.9855)")

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpli9uys1k/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpli9uys1k/assets
/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-08-19 13:44:27.961835: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-19 13:44:27.961845: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-19 13:44:27.962023: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpli9uys1k
2026-08-19 13:44:27.963241: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-19 13:44:27.963245: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77

int8 new-data eval:     0.9917   (fp32: 0.9833)
int8 original test set: 0.9855   (fp32: 0.9855)
